# Final Project Data Science: Prediksi Harga Mobil
**Nama:** Husni Ayi Nurdin | **NPM:** 237006002

**Metodologi:** CRISP-DM | **Model:** Linear Regression

## Langkah 1: Memanggil Library yang Diperlukan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

print('Semua library berhasil diimport!')

## Langkah 2: Load Data

In [ ]:
df = pd.read_csv('Car_sales.csv')
print(f'Data berhasil dimuat! Shape: {df.shape}')

## Langkah 3: Melihat Data

In [ ]:
print('=== 5 Baris Pertama ===')
display(df.head())
print('\n=== Informasi Dataset ===')
df.info()
print('\n=== Statistik Deskriptif ===')
display(df.describe())

## Langkah 4: Menghapus Missing Value

In [ ]:
print('Missing Value sebelum penanganan:')
print(df.isnull().sum())

# Hapus baris dengan terlalu banyak NaN (>50% kolom)
threshold = len(df.columns) * 0.5
df_cleaned = df.dropna(thresh=threshold)
print(f'\nBaris setelah hapus NaN berlebih: {df_cleaned.shape[0]}')

## Langkah 5: Mengisi Missing Value

In [ ]:
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
categorical_cols = df_cleaned.select_dtypes(include=['object']).columns

# Isi numerik dengan median
for col in numeric_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())
        print(f'Kolom {col} diisi median')

# Isi kategorikal dengan modus
for col in categorical_cols:
    if df_cleaned[col].isnull().sum() > 0:
        df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])
        print(f'Kolom {col} diisi modus')

df_cleaned = df_cleaned.dropna(subset=['Price_in_thousands','Sales_in_thousands'])
print('\nMissing Value setelah penanganan:')
print(df_cleaned.isnull().sum())
print(f'Shape akhir: {df_cleaned.shape}')

## Langkah 6: Eksplorasi Data
### 6a. 10 Mobil dengan Penjualan Terbanyak

In [ ]:
top10 = df_cleaned.nlargest(10, 'Sales_in_thousands').reset_index(drop=True)
top10['Car_Name'] = top10['Manufacturer'] + ' ' + top10['Model']

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 1.0, 10))[::-1]
bars = ax.barh(top10['Car_Name'][::-1], top10['Sales_in_thousands'][::-1], color=colors, edgecolor='white')
for bar, val in zip(bars, top10['Sales_in_thousands'][::-1]):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2, f'{val:.1f}k', va='center', fontsize=10, fontweight='bold')
ax.set_title('Top 10 Mobil dengan Penjualan Terbanyak', fontsize=14, fontweight='bold')
ax.set_xlabel('Penjualan (ribuan unit)')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

print(top10[['Car_Name','Sales_in_thousands']].to_string(index=False))

**Analisis:** Mobil dari pabrikan Amerika (Ford, Chevrolet) mendominasi penjualan. Segmen pickup dan sedan keluarga paling laku di pasar. Harga terjangkau menjadi faktor utama tingginya volume penjualan.

### 6b. Harga dari 10 Mobil Terlaris

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors2 = plt.cm.Blues(np.linspace(0.4, 0.9, 10))
bars2 = ax.bar(range(len(top10)), top10['Price_in_thousands'], color=colors2, edgecolor='white', width=0.6)
for bar, val in zip(bars2, top10['Price_in_thousands']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'${val:.1f}k', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(range(len(top10)))
ax.set_xticklabels(top10['Car_Name'], rotation=40, ha='right', fontsize=9)
ax.set_title('Harga dari 10 Mobil dengan Penjualan Terbanyak', fontsize=14, fontweight='bold')
ax.set_ylabel('Harga (ribuan USD)')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

print(top10[['Car_Name','Price_in_thousands']].to_string(index=False))

**Analisis:** Mobil terlaris tidak selalu termahal. Rentang harga $15.000–$35.000 (segmen menengah) mendominasi. Ini membuktikan bahwa harga kompetitif sangat berpengaruh terhadap volume penjualan.

### 6c. Atribut Tambahan dari 10 Mobil Terlaris (Horsepower, Engine Size, Fuel Efficiency)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Atribut Tambahan dari 10 Mobil Terlaris', fontsize=14, fontweight='bold')

# Horsepower
axes[0].bar(range(len(top10)), top10['Horsepower'], color=plt.cm.Oranges(np.linspace(0.4,0.9,10)))
axes[0].set_xticks(range(len(top10)))
axes[0].set_xticklabels(top10['Car_Name'], rotation=45, ha='right', fontsize=8)
axes[0].set_title('Horsepower (HP)'); axes[0].set_ylabel('HP')
for i,v in enumerate(top10['Horsepower']): axes[0].text(i, v+1, f'{v:.0f}', ha='center', fontsize=8)

# Engine Size
axes[1].bar(range(len(top10)), top10['Engine_size'], color=plt.cm.Greens(np.linspace(0.4,0.9,10)))
axes[1].set_xticks(range(len(top10)))
axes[1].set_xticklabels(top10['Car_Name'], rotation=45, ha='right', fontsize=8)
axes[1].set_title('Engine Size (L)'); axes[1].set_ylabel('Liter')
for i,v in enumerate(top10['Engine_size']): axes[1].text(i, v+0.02, f'{v:.1f}L', ha='center', fontsize=8)

# Fuel Efficiency
axes[2].bar(range(len(top10)), top10['Fuel_efficiency'], color=plt.cm.Purples(np.linspace(0.4,0.9,10)))
axes[2].set_xticks(range(len(top10)))
axes[2].set_xticklabels(top10['Car_Name'], rotation=45, ha='right', fontsize=8)
axes[2].set_title('Fuel Efficiency (MPG)'); axes[2].set_ylabel('MPG')
for i,v in enumerate(top10['Fuel_efficiency']): axes[2].text(i, v+0.3, f'{v:.0f}', ha='center', fontsize=8)

for ax in axes: ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

**Analisis:**
- **Horsepower:** Mobil terlaris umumnya bertenaga 120–200 HP, cukup untuk kebutuhan sehari-hari tanpa biaya yang terlalu tinggi.
- **Engine Size:** Mesin 2.0L–4.0L mendominasi. Mesin kecil lebih irit, mesin besar memberikan performa lebih.
- **Fuel Efficiency:** Efisiensi 20–35 MPG menjadi daya tarik konsumen yang mempertimbangkan biaya operasional jangka panjang.

## Langkah 7: Menentukan Variabel Rekomendasi Spesifikasi

**Variabel Independen (X):**
1. `Sales_in_thousands` – popularitas di pasar
2. `__year_resale_value` – nilai jual kembali
3. `Engine_size` – ukuran mesin
4. `Horsepower` – tenaga mesin
5. `Wheelbase` – jarak sumbu roda
6. `Width` – lebar kendaraan
7. `Length` – panjang kendaraan
8. `Curb_weight` – bobot kendaraan
9. `Fuel_capacity` – kapasitas tangki
10. `Fuel_efficiency` – efisiensi BBM
11. `Manufacturer` – merek (one-hot encoded)
12. `Vehicle_type` – jenis kendaraan (one-hot encoded)

**Variabel Dependen (Y):** `Price_in_thousands`

**Rekomendasi Spesifikasi Pasar:** Engine 2.0–3.0L | 130–200 HP | 25–35 MPG | Harga $15.000–$30.000

## Langkah 8: Membuat Model Prediksi dengan Linear Regression
### 8a. Memisahkan Variabel Independent dan Dependent

In [ ]:
df_model = df_cleaned.copy()
df_model = pd.get_dummies(df_model, columns=['Manufacturer','Vehicle_type'], drop_first=False)

feature_cols_base = ['Sales_in_thousands','__year_resale_value','Engine_size','Horsepower',
                     'Wheelbase','Width','Length','Curb_weight','Fuel_capacity','Fuel_efficiency']
ohe_cols = [c for c in df_model.columns if c.startswith('Manufacturer_') or c.startswith('Vehicle_type_')]
feature_cols = [c for c in (feature_cols_base + ohe_cols) if c in df_model.columns]

X = df_model[feature_cols]
y = df_model['Price_in_thousands']
print(f'Jumlah fitur (X): {X.shape[1]}')
print(f'Jumlah sampel   : {X.shape[0]}')
print(f'Target (y)      : Price_in_thousands')

### 8b. Memisahkan Data Training 80% dan Testing 20%

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Data Training : {X_train.shape[0]} sampel')
print(f'Data Testing  : {X_test.shape[0]} sampel')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### 8c. Membuat Model Regresi Linear

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)
print('Model Linear Regression berhasil dilatih!')
print(f'Intercept : {model.intercept_:.4f}')
print(f'Jumlah koefisien: {len(model.coef_)}')

### 8d. Evaluasi Model dengan RMSE dan R2 Score

In [ ]:
y_pred_train = model.predict(X_train_scaled)
y_pred_test  = model.predict(X_test_scaled)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test, y_pred_test)

print('='*45)
print(f'{"HASIL EVALUASI MODEL LINEAR REGRESSION":^45}')
print('='*45)
print(f'{"Metrik":<20}{"Training":>12}{"Testing":>12}')
print(f'{"-"*44}')
print(f'{"RMSE":<20}{rmse_train:>12.4f}{rmse_test:>12.4f}')
print(f'{"R2 Score":<20}{r2_train:>12.4f}{r2_test:>12.4f}')
print('='*45)

### 8e. Scatter Plot Evaluasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Evaluasi Model Linear Regression - Harga Aktual vs Prediksi', fontsize=14, fontweight='bold')

# Training
axes[0].scatter(y_train, y_pred_train, alpha=0.6, color='steelblue', edgecolors='white', s=60)
mn, mx = min(y_train.min(), y_pred_train.min()), max(y_train.max(), y_pred_train.max())
axes[0].plot([mn,mx],[mn,mx],'r--',lw=2,label='Perfect Prediction')
axes[0].set_title(f'Data Training\nR2={r2_train:.4f} | RMSE={rmse_train:.4f}', fontweight='bold')
axes[0].set_xlabel('Harga Aktual (thousands USD)')
axes[0].set_ylabel('Harga Prediksi (thousands USD)')
axes[0].legend()

# Testing
axes[1].scatter(y_test, y_pred_test, alpha=0.7, color='#e67e22', edgecolors='white', s=60)
mn2, mx2 = min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())
axes[1].plot([mn2,mx2],[mn2,mx2],'r--',lw=2,label='Perfect Prediction')
axes[1].set_title(f'Data Testing\nR2={r2_test:.4f} | RMSE={rmse_test:.4f}', fontweight='bold')
axes[1].set_xlabel('Harga Aktual (thousands USD)')
axes[1].set_ylabel('Harga Prediksi (thousands USD)')
axes[1].legend()

plt.tight_layout(); plt.show()
print('Titik yang rapat di sekitar garis merah = prediksi akurat!')

**Interpretasi Scatter Plot:** Titik-titik yang mendekati garis merah putus-putus (perfect prediction line y=x) menunjukkan model dapat memprediksi harga dengan baik. Semakin rapat titik di sekitar garis, semakin tinggi akurasi model.

## Langkah 9: Memprediksi Harga dengan Spesifikasi yang Ditentukan

In [ ]:
# Spesifikasi mobil rekomendasi berdasarkan analisis pasar
spec_input = {
    'Manufacturer': 'Ford', 'Vehicle_type': 'Passenger',
    'Sales_in_thousands': 50.0, '__year_resale_value': 18.0,
    'Engine_size': 2.5, 'Horsepower': 150.0,
    'Wheelbase': 105.0, 'Width': 68.0, 'Length': 175.0,
    'Curb_weight': 3.0, 'Fuel_capacity': 14.5, 'Fuel_efficiency': 28.0
}

df_spec = pd.DataFrame([spec_input])
print('Spesifikasi Input:')
display(df_spec)

# Buat dataframe dengan semua fitur model
df_pred = pd.DataFrame(np.zeros((1, len(feature_cols))), columns=feature_cols)
for col in feature_cols_base:
    if col in df_spec.columns:
        df_pred.at[0, col] = float(df_spec[col].values[0])

manuf_col = f"Manufacturer_{spec_input['Manufacturer']}"
if manuf_col in df_pred.columns: df_pred.at[0, manuf_col] = 1.0

vtype_col = f"Vehicle_type_{spec_input['Vehicle_type']}"
if vtype_col in df_pred.columns: df_pred.at[0, vtype_col] = 1.0

df_pred_scaled = scaler.transform(df_pred)
predicted_price = model.predict(df_pred_scaled)[0]

## Langkah 10: Menampilkan Hasil Prediksi

In [ ]:
print('='*55)
print(f'{"HASIL PREDIKSI HARGA MOBIL":^55}')
print('='*55)
print(f'  Manufacturer   : {spec_input["Manufacturer"]}')
print(f'  Vehicle Type   : {spec_input["Vehicle_type"]}')
print(f'  Engine Size    : {spec_input["Engine_size"]} L')
print(f'  Horsepower     : {spec_input["Horsepower"]} HP')
print(f'  Fuel Effic.    : {spec_input["Fuel_efficiency"]} MPG')
print('-'*55)
print(f'  PREDIKSI HARGA : ${predicted_price:.2f} ribu (${predicted_price*1000:,.2f})')
print('='*55)